In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import platform
from matplotlib import rc

if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
else:
    rc('font', family='Malgun Gothic')

from IPython.display import Image

np.set_printoptions(suppress=True, precision=4)
pd.options.display.float_format = '{:,.4f}'.format
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
plt.rcParams['font.size'] = 14

random_seed = 123

In [ ]:
# CSV 파일 읽기
df = pd.read_csv('input/한국철도공사_시군구단위 거주 철도이용객 이용 시간대별 소비업종 정보_20221115.csv')

print("=" * 60)
print("데이터 기본 정보")
print("=" * 60)
print(f"데이터 크기: {df.shape[0]:,}행 x {df.shape[1]:}열")
print(f"\n컬럼명:")
for i, col in enumerate(df.columns):
    print(f"  {i}: {col}")

In [ ]:
# 결측치 확인
print("=" * 60)
print("결측치 확인")
print("=" * 60)
print(df.isnull().sum())
print(f"\n결측치 비율: {(df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100):.2f}%")

In [ ]:
# 기본 통계
print("=" * 60)
print("기본 통계")
print("=" * 60)
print(df.describe())

In [ ]:
# 카테고리 데이터 확인
print("=" * 60)
print("카테고리 데이터 확인")
print("=" * 60)
print(f"시도 종류: {df['시도명'].nunique()}개")
print(f"시도 목록: {sorted(df['시도명'].unique())}")
print(f"\n시군구 종류: {df['시군구명'].nunique()}개")
print(f"\n시간대 코드: {sorted(df['시간코드'].unique())}")
print(f"시간대 종류: {df['시간코드'].nunique()}개")

# 수치형 컬럼만 추출
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\n수치형 컬럼 ({len(numeric_cols)}개):")
for col in numeric_cols:
    print(f"  - {col}")

In [ ]:
# 데이터 상관관계 확인
print("=" * 60)
print("수치형 데이터 상관관계")
print("=" * 60)
correlation = df[numeric_cols].corr()
print(correlation['전체_카드'].sort_values(ascending=False))

In [ ]:
# 시간대별, 시도별 평균 소비액
print("=" * 60)
print("시간대별 평균 소비액")
print("=" * 60)
print(df.groupby('시간코드')['전체_카드'].agg(['count', 'mean', 'std', 'min', 'max']))

print("\n" + "=" * 60)
print("시도별 평균 소비액")
print("=" * 60)
print(df.groupby('시도명')['전체_카드'].agg(['count', 'mean', 'std', 'min', 'max']).sort_values('mean', ascending=False))

In [ ]:
# 데이터 분포 시각화
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 전체_카드 분포
axes[0, 0].hist(df['전체_카드'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('전체_카드 분포')
axes[0, 0].set_xlabel('카드 사용액')
axes[0, 0].set_ylabel('빈도')

# 2. 시간대별 평균 소비액
time_avg = df.groupby('시간코드')['전체_카드'].mean()
axes[0, 1].bar(time_avg.index, time_avg.values, color='steelblue', edgecolor='black')
axes[0, 1].set_title('시간대별 평균 소비액')
axes[0, 1].set_xlabel('시간코드')
axes[0, 1].set_ylabel('평균 카드 사용액')

# 3. 시도별 평균 소비액 TOP 10
province_avg = df.groupby('시도명')['전체_카드'].mean().sort_values(ascending=False).head(10)
axes[1, 0].barh(range(len(province_avg)), province_avg.values, color='coral', edgecolor='black')
axes[1, 0].set_yticks(range(len(province_avg)))
axes[1, 0].set_yticklabels(province_avg.index)
axes[1, 0].set_title('시도별 평균 소비액 TOP 10')
axes[1, 0].set_xlabel('평균 카드 사용액')

# 4. 소비 업종별 분포
spending_cols = [col for col in numeric_cols if col != '전체_카드' and col not in ['법정동시도코드', '법정동시군구코드', '시간코드']]
spending_sum = df[spending_cols].sum().sort_values(ascending=False)
axes[1, 1].bar(range(len(spending_sum)), spending_sum.values, color='mediumpurple', edgecolor='black')
axes[1, 1].set_xticks(range(len(spending_sum)))
axes[1, 1].set_xticklabels(spending_sum.index, rotation=45, ha='right')
axes[1, 1].set_title('소비 업종별 총액')
axes[1, 1].set_ylabel('총 카드 사용액')

plt.tight_layout()
plt.savefig('output/01_데이터분포분석.png', dpi=100, bbox_inches='tight')
plt.show()

print("그래프 저장 완료: output/01_데이터분포분석.png")